# Mechanisms in BrainCell

This notebook is the mechanism-layer sequel to `2.cell.ipynb`.

There, `paint(...)` and `place(...)` were introduced as declarative APIs. Here we zoom in on **what those declarations actually are** and how they behave at runtime in the current multi-compartment stack.

At the declaration layer, `braincell.mech` is organized around two mechanism families:

- `Density`: region-based declarations installed with `cell.paint(...)`
- `Point`: point-located declarations installed with `cell.place(...)`

`Mechanism` is the common marker base class for both families. `CableProperty` belongs next to this taxonomy because it is also painted onto regions, but it is a passive cable declaration rather than a `Mechanism` subclass.

This notebook fully runs through density mechanisms, clamps, and probes. It only summarizes `ProbeMechanism`, `Synapse`, and `Junction`, because the current multi-compartment tutorial surface for those declarations is still narrower.

| Family | Declaration | Attach with | Purpose | Coverage here |
| --- | --- | --- | --- | --- |
| Related passive declaration | `CableProperty` | `cell.paint(region, ...)` | Set passive cable defaults such as `v_rest`, `cm`, `ra`, and `temperature` | Runnable below |
| Density mech | `Ion` | `cell.paint(region, ...)` | Declare ion species containers and ion parameters | Runnable below |
| Density mech | `Channel` | `cell.paint(region, ...)` | Declare distributed ion-channel mechanisms | Runnable below |
| Point mech | `CurrentClamp` | `cell.place(locset, ...)` | Piecewise-constant current injection | Runnable below |
| Point mech | `SineClamp` | `cell.place(locset, ...)` | Sinusoidal current injection | Runnable below |
| Point mech | `FunctionClamp` | `cell.place(locset, ...)` | Arbitrary callable current injection | Runnable below |
| Point mech | `StateProbe` | `cell.place(locset, ...)` | Probe cell-owned state such as `v` | Runnable below |
| Point mech | `MechanismProbe` | `cell.place(locset, ...)` | Probe a named runtime field on a mechanism | Runnable below |
| Point mech | `CurrentProbe` | `cell.place(locset, ...)` | Probe current from a mechanism or ion owner | Runnable below |
| Point mech | `ProbeMechanism` | `cell.place(locset, ...)` | Legacy variable-by-name recorder | Overview only |
| Point mech | `Synapse` | `cell.place(locset, ...)` | Registry-keyed synapse declaration | Overview only |
| Point mech | `Junction` | `cell.place(locset, ...)` | Gap-junction declaration placeholder | Overview only |


In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import brainunit as u
import braincell
import braincell.channel
import braincell.ion
import braincell.synapse
from braincell import Branch, Cell, Morphology
from braincell.filter import AllRegion, BranchSlice, RootLocation, at
from braincell.mech import (
    CableProperty,
    Channel,
    CurrentClamp,
    CurrentProbe,
    FunctionClamp,
    Ion,
    MechanismProbe,
    SineClamp,
    StateProbe,
)


def build_demo_morphology() -> Morphology:
    soma = Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[8.0, 8.0] * u.um,
        type="soma",
    )
    dend = Branch.from_lengths(
        lengths=[60.0] * u.um,
        radii=[2.0, 1.2] * u.um,
        type="basal_dendrite",
    )
    axon = Branch.from_lengths(
        lengths=[80.0] * u.um,
        radii=[1.0, 0.6] * u.um,
        type="axon",
    )
    morpho = Morphology.from_root(soma, name="soma")
    morpho.attach(parent="soma", child_branch=dend, child_name="dend", parent_x=1.0)
    morpho.attach(parent="soma", child_branch=axon, child_name="axon", parent_x=0.0)
    return morpho


ERROR:2026-04-23 10:48:22,964:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 257, in _check_cuda_versions
    cublas_version = _version_check("cuBLAS", cuda_versions.cublas_get_version,
  File "/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 217, in _version_check
    raise RuntimeError(msg)
RuntimeError: Outdated cuBLAS installation found.
Version JAX was built against: 12

## 1. Density mechanisms

We start with the declarations that are painted over a region of the cell.

In practice, the density story has three layers:

1. `CableProperty` sets passive cable defaults.
2. `Ion` creates or customizes ion containers.
3. `Channel` installs distributed conductance mechanisms that bind to those ions.


### 1.1 `CableProperty`: passive defaults for a region

`CableProperty` is the passive, non-`Mechanism` companion to density declarations.
It uses the same `paint(...)` entry point, so it is worth seeing first.


In [2]:
morpho = build_demo_morphology()
print(morpho.topo())

cell = Cell(morpho)
cell.paint(
    AllRegion(),
    CableProperty(
        resting_potential=-65.0 * u.mV,
        membrane_capacitance=1.0 * (u.uF / u.cm**2),
        axial_resistivity=100.0 * (u.ohm * u.cm),
    ),
)

print()
print(cell)
print("root CV resting potential:", cell.cvs[0].v)


soma
├── dend
└── axon

Cell(root='soma', n_branches=3, n_paint_rules=1, n_place_rules=0, initialized=False)
root CV resting potential: -65. mV


### 1.2 `Channel` with fallback ions: automatic binding

A fresh `Cell` still has fallback ion containers for `na`, `k`, and `ca`.
If a channel needs one ion family and that family is still unambiguous, the channel can bind automatically without any selector.


In [3]:
cell_auto = Cell(build_demo_morphology())
cell_auto.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Channel("Na_HH1952", g_max=12.0 * (u.mS / u.cm**2)),
)
cell_auto.init_state()

layout = next(layout for layout in cell_auto.layouts if layout.kind == "channel:Na_HH1952")
node = cell_auto.get_runtime_node(layout.id)
na = cell_auto.get_ion("na")

print(type(na).__name__)
print(type(node).__name__)
print("bound to fallback na:", na.channels["Na_HH1952"] is node)


SodiumFixed
Na_HH1952
bound to fallback na: True


### 1.3 `Ion` + `Channel`: explicit binding when one family is ambiguous

Once one ion family has multiple runtime instances, family-level lookup becomes ambiguous.
That is the moment to use `ion_name="..."` on a single-ion channel.


In [4]:
cell_named = Cell(build_demo_morphology())
cell_named.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Ion("SodiumFixed", name="na_soma", E=55.0 * u.mV),
)
cell_named.paint(
    BranchSlice(branch_index=1, prox=0.0, dist=1.0),
    Ion("SodiumFixed", name="na_dend", E=45.0 * u.mV),
)
cell_named.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Channel("Na_HH1952", ion_name="na_soma", g_max=12.0 * (u.mS / u.cm**2)),
)
cell_named.init_state()

print("layout kinds:", [layout.kind for layout in cell_named.layouts])
print("na_soma name:", cell_named.get_ion("na_soma").name)
print("na_soma E shape:", cell_named.get_ion("na_soma").E.shape)

try:
    cell_named.get_ion("na")
except Exception as exc:
    print(type(exc).__name__, exc)


layout kinds: ['ion:SodiumFixed', 'channel:Na_HH1952', 'ion:SodiumFixed']
na_soma name: na_soma
na_soma E shape: (7,)
ValueError Ion selector 'na' is ambiguous; family 'na' has candidates ['na_soma', 'na_dend'].


### 1.4 Mixed-ion `Channel`: current owner versus modulators

Some channels depend on more than one ion family.
`Kca3p1_MA2020` uses potassium as the current owner, while calcium modulates gating.

For mixed-ion channels the selector is `ion_names={...}` and it only needs to name the ambiguous family.


In [5]:
cell_mixed = Cell(build_demo_morphology())
cell_mixed.paint(
    BranchSlice(branch_index=[0, 1], prox=0.0, dist=1.0),
    Ion("PotassiumFixed", name="k_main", E=-88.0 * u.mV),
    Ion("CalciumFixed", name="ca_hva", Ci=2e-4 * u.mM),
    Ion("CalciumFixed", name="ca_lva", Ci=5e-4 * u.mM),
)
cell_mixed.paint(
    BranchSlice(branch_index=[0, 1], prox=0.0, dist=1.0),
    Channel("Kca3p1_MA2020", ion_names={"ca": "ca_hva"}),
)
cell_mixed.place(
    at("soma", 0.5),
    CurrentProbe(mechanism="Kca3p1_MA2020"),
    CurrentProbe(ion="k_main"),
)
cell_mixed.init_state()

samples = cell_mixed.sample_probes()
layout = next(layout for layout in cell_mixed.layouts if layout.kind == "channel:Kca3p1_MA2020")

print(sorted(samples))
print("bound ion keys:", cell_mixed.runtime.bound_ion_keys[layout.id])
print("current owner:", cell_mixed.runtime.current_owner_keys[layout.id])
print("owner has channel:", "Kca3p1_MA2020" in cell_mixed.get_ion("k_main").channels)
print("modulator has channel:", "Kca3p1_MA2020" in cell_mixed.get_ion("ca_hva").channels)


['soma(0.5)_Kca3p1_MA2020_current', 'soma(0.5)_k_main_current']
bound ion keys: ('k_main', 'ca_hva')
current owner: k_main
owner has channel: True
modulator has channel: False


## 2. Point mechanisms

Point mechanisms live at one location in the point tree instead of being painted across a region.

The current point-mechanism surface falls into two practical groups:

- **stimuli**: clamps that inject current at one point
- **observers**: probes that read back runtime state or current at one point


### 2.1 Clamp declarations: `CurrentClamp`, `SineClamp`, `FunctionClamp`

These three declarations all contribute through the same runtime path: `cell.runtime.evaluate_point_clamps(t=...)`.
The point current at one location is the sum of every active clamp placed there.


In [6]:
cell_clamp = Cell(build_demo_morphology())
cell_clamp.place(
    RootLocation(x=0.5),
    CurrentClamp(
        start=0.0 * u.ms,
        durations=(2.0 * u.ms, 2.0 * u.ms),
        amplitudes=(0.0 * u.nA, 0.3 * u.nA),
    ),
    SineClamp(
        amplitude=0.2 * u.nA,
        frequency=500.0 * u.Hz,
        offset=0.1 * u.nA,
        duration=4.0 * u.ms,
    ),
    FunctionClamp(
        fn=lambda local_t: 0.4 * u.nA if local_t < 1.0 * u.ms else 0.0 * u.nA,
        duration=3.0 * u.ms,
    ),
)
cell_clamp.init_state()

point_index = int(cell_clamp.layouts[0].point_index[0])
current_early = cell_clamp.runtime.evaluate_point_clamps(t=0.5 * u.ms)
current_late = cell_clamp.runtime.evaluate_point_clamps(t=2.5 * u.ms)

print("layout kinds:", [layout.kind for layout in cell_clamp.layouts])
print("active point index:", point_index)
print(f"combined clamp current at 0.5 ms: {float(current_early[point_index].to_decimal(u.nA)):.3f} nA")
print(f"combined clamp current at 2.5 ms: {float(current_late[point_index].to_decimal(u.nA)):.3f} nA")


layout kinds: ['CurrentClamp', 'SineClamp', 'FunctionClamp']
active point index: 1
combined clamp current at 0.5 ms: 0.700 nA
combined clamp current at 2.5 ms: 0.600 nA


### 2.2 Probe declarations: `StateProbe`, `MechanismProbe`, `CurrentProbe`

Probes are sparse point declarations. They do not allocate their own evolving state; they sample existing runtime values by explicit selector.


In [7]:
cell_probe = Cell(build_demo_morphology())
cell_probe.paint(
    BranchSlice(branch_index=[0, 1], prox=0.0, dist=1.0),
    Channel(
        "Na_HH1952",
        g_max=12.0 * (u.mS / u.cm**2),
        V_sh=-50.0 * u.mV,
        temp=u.celsius2kelvin(36.0),
    ),
)
cell_probe.place(
    at("soma", 0.5),
    StateProbe(),
    MechanismProbe(mechanism="Na_HH1952", field="p"),
    CurrentProbe(ion="na", mechanism="Na_HH1952"),
)
cell_probe.init_state()

samples = cell_probe.sample_probes()

print("probe keys:", sorted(samples))
print("soma(0.5)_v (mV):", float(samples["soma(0.5)_v"].to_decimal(u.mV)))
print("soma(0.5)_Na_HH1952_p:", float(samples["soma(0.5)_Na_HH1952_p"]))
print(
    "soma(0.5)_Na_HH1952_current (mA / cm^2):",
    float(samples["soma(0.5)_Na_HH1952_current"].to_decimal(u.mA / u.cm**2)),
)


probe keys: ['soma(0.5)_Na_HH1952_current', 'soma(0.5)_Na_HH1952_p', 'soma(0.5)_v']
soma(0.5)_v (mV): -65.0
soma(0.5)_Na_HH1952_p: 0.0
soma(0.5)_Na_HH1952_current (mA / cm^2): 0.0


### 2.3 Other point declarations

Three public point declarations are worth knowing even though we do not build a full end-to-end example for them here.

- `ProbeMechanism(variable="v", target="soma")` is the older variable-by-name recorder. In new multi-compartment code, prefer `StateProbe`, `MechanismProbe`, and `CurrentProbe` when you want explicit selectors.
- `Synapse(synapse_type="AMPA", ...)` is the registry-keyed point declaration for synaptic mechanisms. The declaration surface exists, but this notebook does not build an event-driven synaptic input pipeline.
- `Junction(params={"g": 1.0})` is currently a placeholder for future gap-junction wiring. It stores parameters, but partner wiring is not yet part of the public multi-cell story.


## Summary

You have now seen the current mechanism surface in the multi-compartment stack:

- `CableProperty` is paintable and central to the workflow, but it sits beside `Mechanism`, not under it.
- `Ion` and `Channel` are the two concrete density-mechanism declarations.
- Automatic channel binding only works while each required ion family is still unambiguous.
- `CurrentClamp`, `SineClamp`, and `FunctionClamp` all feed the same point-current evaluation path.
- `StateProbe`, `MechanismProbe`, and `CurrentProbe` let you read back runtime values without creating their own evolving state.
- `ProbeMechanism`, `Synapse`, and `Junction` are part of the public declaration surface, but deserve separate tutorials once their broader workflows are shown end to end.
